# STG 4 — Features semánticas de títulos de proyectos

**Objetivo**: convertir cada `titulo_base` en un vector numérico que capture su significado
y agrupar los títulos en temas interpretables.

**Entrada**: `data/df_modelado.csv`  
**Salida**: `data/df_features_titulo.csv`

**Flujo del notebook**:
1. Cargar títulos únicos
2. Calcular embeddings (vectores semánticos)
3. Explorar agrupamientos con K=10, 15, 20
4. ⚠️ **CHECKPOINT HUMANO**: elegir K
5. Generar resultado final y guardar

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

df = pd.read_csv('../data/df_modelado.csv')
titulos = df['titulo_base'].dropna().unique()
titulos = pd.Series(titulos).reset_index(drop=True)

print(f'Títulos únicos en df_modelado: {len(titulos)}')
titulos.head(10)

## Paso 1 — Embeddings

Cada título se convierte en un vector de 384 números que captura su significado semántico.
El modelo `paraphrase-multilingual-MiniLM-L12-v2` fue entrenado en múltiples idiomas
incluyendo español. La primera vez descarga el modelo (~120 MB); las siguientes lo usa desde caché.

In [ ]:
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print('Calculando embeddings...')
embeddings = modelo.encode(titulos.tolist(), show_progress_bar=True)

print(f'\nShape de la matriz de embeddings: {embeddings.shape}')
print(f'  → {embeddings.shape[0]} títulos × {embeddings.shape[1]} dimensiones')

## Paso 2 — Exploración de agrupamientos (K=10, 15, 20)

Probamos tres valores de K. Para cada uno se muestra el tamaño de cada grupo y los 5 títulos
más representativos (los más cercanos al centro del grupo).

⚠️ **Después de revisar esta celda hay un checkpoint: elegís el K que produce grupos más coherentes.**

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

# Palabras a ignorar al generar etiquetas (stopwords + términos legislativos vacíos)
STOPWORDS = {
    'de', 'la', 'el', 'los', 'las', 'del', 'en', 'y', 'a', 'por', 'para',
    'con', 'se', 'su', 'al', 'un', 'una', 'que', 'o', 'e', 'es', 'no',
    'lo', 'le', 'sus', 'entre', 'sobre', 'como', 'más', 'si', 'ha', 'pero',
    # términos legislativos sin valor semántico
    'ley', 'leyes', 'nacional', 'nacionales', 'artículo', 'articulo',
    'modificación', 'modificacion', 'modif', 'proyecto', 'establece',
    'establecimiento', 'creación', 'creacion', 'regimen', 'régimen',
    'república', 'republica', 'argentina', 'honorable', 'cámara', 'camara',
    'diputados', 'senado', 'nación', 'nacion', 'poder', 'ejecutivo',
}

def etiqueta_grupo(titulos_grupo):
    """Devuelve las 5 palabras más frecuentes del grupo (excluyendo stopwords)."""
    palabras = []
    for t in titulos_grupo:
        tokens = re.findall(r'\b[a-záéíóúüñ]{4,}\b', t.lower())
        palabras.extend([p for p in tokens if p not in STOPWORDS])
    from collections import Counter
    top = Counter(palabras).most_common(5)
    return ' / '.join(w for w, _ in top)


def explorar_k(k, embeddings, titulos):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)
    centros = km.cluster_centers_

    print(f'\n{"="*60}')
    print(f'  K = {k}')
    print(f'{"="*60}')

    for grupo_id in range(k):
        idx_grupo = np.where(labels == grupo_id)[0]
        titulos_grupo = titulos.iloc[idx_grupo].tolist()

        # Los 5 más cercanos al centro
        distancias = cosine_distances([centros[grupo_id]], embeddings[idx_grupo])[0]
        top5_idx = np.argsort(distancias)[:5]
        representativos = [titulos_grupo[i] for i in top5_idx]

        etiqueta = etiqueta_grupo(titulos_grupo)
        print(f'\n  Grupo {grupo_id:>2} ({len(idx_grupo):>3} títulos) | {etiqueta}')
        for t in representativos:
            print(f'    · {t[:80]}')

    return labels

for k in [10, 15, 20]:
    explorar_k(k, embeddings, titulos)

## ⚠️ CHECKPOINT — Elección de K

El equipo revisó los agrupamientos para K=10, 15 y 20.
**K elegido: 20** — produce grupos más específicos y coherentes temáticamente.

La celda siguiente fija K=20, recalcula el clustering final y genera el diccionario
de clasificaciones (grupo_id → etiqueta) que queda documentado en el notebook.

In [ ]:
# ── Clustering final ──────────────────────────────────────────────────────────
K_ELEGIDO = 20  # decisión del equipo tras revisar K=10, 15, 20

km_final = KMeans(n_clusters=K_ELEGIDO, random_state=42, n_init=10)
labels_final = km_final.fit_predict(embeddings)
centros_final = km_final.cluster_centers_

# ── Grupos sin tema ───────────────────────────────────────────────────────────
# Grupo 11: mayoría de títulos son "Votación en General y Particular de Proyectos..."
# → votos sobre lote de proyectos sin ley específica identificable.
# Votos reales pero el título no tiene contenido temático propio.
# Nota: hardcodeado porque el clustering es determinista (random_state=42, datos fijos).
GRUPOS_SIN_TEMA = {11}

# ── Generar etiquetas automáticas ─────────────────────────────────────────────
mapa_grupos = {}

for grupo_id in range(K_ELEGIDO):
    idx = np.where(labels_final == grupo_id)[0]
    titulos_grupo = titulos.iloc[idx].tolist()

    if grupo_id in GRUPOS_SIN_TEMA:
        label = 'sin_clasificar'
    else:
        label = etiqueta_grupo(titulos_grupo)

    dist = cosine_distances([centros_final[grupo_id]], embeddings[idx])[0]
    top3 = [titulos_grupo[i] for i in np.argsort(dist)[:3]]

    mapa_grupos[grupo_id] = {'label': label, 'n': len(idx), 'ejemplos': top3}

# ── Tabla de clasificaciones ──────────────────────────────────────────────────
print('TABLA DE CLASIFICACIONES — K=20')
print('=' * 75)
print(f'{"ID":>3}  {"ETIQUETA":<42}  {"TÍTULOS":>7}  NOTA')
print('-' * 75)
for gid, info in mapa_grupos.items():
    nota = ' ← sin tema (se conservan)' if gid in GRUPOS_SIN_TEMA else ''
    ejemplo = info['ejemplos'][0][:35] if info['ejemplos'] else ''
    print(f'{gid:>3}  {info["label"]:<42}  {info["n"]:>7}  {ejemplo}{nota}')

n_sin_tema  = sum(info['n'] for gid, info in mapa_grupos.items() if gid in GRUPOS_SIN_TEMA)
n_con_tema  = sum(info['n'] for gid, info in mapa_grupos.items() if gid not in GRUPOS_SIN_TEMA)
print(f'\nTítulos con tema asignado:   {n_con_tema}')
print(f'Títulos sin_clasificar:      {n_sin_tema}')
print(f'Total:                       {len(titulos)}')

## Paso 3 — Armar df_features_titulo y guardar

Construye una fila por título único con:
- `titulo_base`: el texto del título
- `tema_id`: número del grupo (0-19)
- `tema_label`: etiqueta automática del grupo
- `emb_0` ... `emb_383`: el vector semántico (384 dimensiones)

Se guarda en `data/df_features_titulo.csv`. No toca `df_modelado.csv`.

In [ ]:
# Armar columnas de tema
tema_ids    = [int(labels_final[i]) for i in range(len(titulos))]
tema_labels = [mapa_grupos[tid]['label'] for tid in tema_ids]

# Armar DataFrame: titulo_base + tema_id + tema_label + embeddings
cols_emb = [f'emb_{i}' for i in range(embeddings.shape[1])]
df_emb = pd.DataFrame(embeddings, columns=cols_emb)

df_features_titulo = pd.concat([
    pd.DataFrame({'titulo_base': titulos, 'tema_id': tema_ids, 'tema_label': tema_labels}),
    df_emb
], axis=1)

# Guardar
ruta_salida = '../data/df_features_titulo.csv'
df_features_titulo.to_csv(ruta_salida, index=False)

# Verificar que df_modelado.csv no fue tocado
df_modelado_check = pd.read_csv('../data/df_modelado.csv')
assert 'titulo_base' in df_modelado_check.columns

print('─' * 50)
print('RESUMEN df_features_titulo')
print('─' * 50)
print(f'Filas (títulos únicos):  {len(df_features_titulo)}')
print(f'Columnas totales:        {len(df_features_titulo.columns)}  (titulo_base + tema_id + tema_label + 384 emb)')
print(f'Columnas de embedding:   emb_0 … emb_{embeddings.shape[1]-1}')
print(f'Temas distintos:         {df_features_titulo["tema_id"].nunique()}')
print(f'Guardado en:             {ruta_salida}')
print()
print('✓ df_modelado.csv intacto.')

df_features_titulo[['titulo_base', 'tema_id', 'tema_label']].head()

## Tabla de validación — 5 títulos al azar por tema

Revisá que los títulos de cada grupo sean coherentes temáticamente.

In [ ]:
np.random.seed(42)

print('VALIDACIÓN — 5 títulos al azar por tema (K=20)')
print('=' * 80)

for gid in range(K_ELEGIDO):
    info = mapa_grupos[gid]
    idx_grupo = np.where(labels_final == gid)[0]
    muestra_idx = np.random.choice(idx_grupo, size=min(5, len(idx_grupo)), replace=False)
    muestra = titulos.iloc[muestra_idx].tolist()

    nota = '  ← sin tema (sin_clasificar)' if gid == GRUPO_SIN_TEMA else ''
    print(f'\nGrupo {gid:>2} | {info["label"]}{nota}  ({info["n"]} títulos)')
    print('-' * 70)
    for t in muestra:
        print(f'  · {t[:75]}')